In [1]:
using LinearAlgebra, Distributions, Random, Plots, LaTeXStrings, DataFrames, CSV

In [2]:
include(joinpath(@__DIR__, "..", "..", "AuxiliaryFunctions.jl"))
include(joinpath(@__DIR__, "..", "..", "SpikeEstimation.jl"))
figdir = "Figures"
tbdir = "Tables"

"Tables"

In [3]:
function MP(x::Float64,d::Float64,dm::Float64,dp::Float64,σ)
    if dm<x<dp
        return (1/(2*π*x*σ*min(d,1)))*sqrt(x-dm)*sqrt(dp-x)
    else
        return 0
    end
end

MP (generic function with 1 method)

In [ ]:
Random.seed!(1234)

N = 5000
σvec = 1.5*ones(N)
σvec[1:3] .= (4.5, 5.0, 5.0)
sqrtΣ = Diagonal(sqrt.(σvec))
dvec = [0.1,0.5,0.9]

W = Matrix{Float64}(undef,N,N)

for m=1:3
    d = dvec[m]
    M = convert(Int64,ceil(N/d))
    X = randn(N,M)
    Y = sqrtΣ*X/sqrt(M)
    mul!(W,Y,Y')
    evals = eigvals(Symmetric(W))

    results = AsympSCM(W, vecNbr=100, nx=1000, tol=3/sqrt(N))
    xvec, yvec, SpikeNbr, SpikeLoc, γmin, γplus = 
        results[:x], results[:density], results[:spikes_nbr], results[:spikes_loc], results[:γmin], results[:γplus]

    true_γplus = 1.5*(1+sqrt(d))^2
    true_γmin = 1.5*(1-sqrt(d))^2
    true_spikes = evals[end-2:end]
    true_yvec = map(x->MP(x,d,true_γmin,true_γplus,1.5),xvec)
    
    p = histogram(evals,bins=evals[1]-0.2:0.1:evals[end]+0.2,normalize=:pdf,label="ESD of "*L"W", legendfontsize=12, framestyle=:box, xtickfontsize=12, ytickfontsize=12,legend=:topright)
    color = theme_palette(:auto).colors[1]
    p = plot!(xvec,yvec,linecolor=:red,linewidth=4,label="Estimated Density")
    p = plot!(xvec,true_yvec,linecolor=:blue,linewidth=2,linestyle=:dash,label="True Density")
    p = scatter!(true_spikes,0*true_spikes,markersize=8,color=color,marker=:dot,label="True Spikes")
    p = scatter!(SpikeLoc,0*SpikeLoc,markersize=5,color=:red,marker=:dot,label="Estimated Spikes")
    dlabel = lpad(Int(round(10d)), 2, '0')
    savefig(p,joinpath(figdir, "Density$(dlabel).pdf"))

    tb = DataFrame(A=true_spikes,B=SpikeLoc,C=abs.(true_spikes-SpikeLoc))
    CSV.write(joinpath(tbdir, "Spikes$(dlabel).csv"),tb)
end

┌ Warning: Keyword argument markerstrokestyle not supported with Plots.GRBackend().  Choose from: annotationcolor, annotationfontfamily, annotationfontsize, annotationhalign, annotationrotation, annotations, annotationvalign, arrow, aspect_ratio, axis, background_color, background_color_inside, background_color_outside, background_color_subplot, bar_width, bins, bottom_margin, camera, clims, color_palette, colorbar, colorbar_entry, colorbar_scale, colorbar_title, colorbar_titlefont, colorbar_titlefontcolor, colorbar_titlefontrotation, colorbar_titlefontsize, connections, contour_labels, discrete_values, fill, fill_z, fillalpha, fillcolor, fillrange, fillstyle, flip, fontfamily, fontfamily_subplot, foreground_color, foreground_color_axis, foreground_color_border, foreground_color_grid, foreground_color_subplot, foreground_color_text, formatter, framestyle, grid, gridalpha, gridlinewidth, gridstyle, group, guide, guidefont, guidefontcolor, guidefontfamily, guidefonthalign, guidefontrotat

In [ ]:
Random.seed!(1234)

Nvec = vcat(200:200:3000,3500:500:8000)
len_N = length(Nvec)
dvec = [0.1,0.5,0.9]
len_d = length(dvec)

SampleNbr = 50
Percent = zeros(Float64,len_d,len_N)
SpikeNbr = zeros(Float64,len_d,len_N)
LanTime = zeros(Float64,len_d,len_N)
EigTime = zeros(Float64,len_d,len_N)

function effprocess!(W)
    N = size(W,1)

    t_eig = @elapsed begin
        evals = eigvals(Symmetric(W))
    end

    results = nothing
    t_lan = @elapsed begin
        results = AsympSCM(W, vecNbr=1,
            tol = 3/sqrt(N),
            compute_density = false)
    end

    return t_eig, t_lan, results[:spikes_nbr]
end

N0 = first(Nvec)
d0 = first(dvec)
M0 = convert(Int64,ceil(N0/d0))
σ0 = 1.5*ones(N0)
σ0[1:3] .= (4.5, 5.0, 5.0)
sqrtΣ0 = Diagonal(sqrt.(σ0))
X0 = randn(N0, M0)
W0 = Matrix{Float64}(undef, N0, N0)
mul!(X0, sqrtΣ0, X0)
X0 .= X0 / sqrt(M0)
mul!(W0, X0, X0')
effprocess!(W0)

for m in 1:len_d
    d = dvec[m]
    for ℓ in 1:len_N
        N = Nvec[ℓ]
        M = convert(Int64,ceil(N/d))
        σvec = 1.5*ones(N)
        σvec[1:3] .= (4.5, 5.0, 5.0)
        sqrtΣ = Diagonal(sqrt.(σvec))

        X = randn(N, M)
        W = Matrix{Float64}(undef, N, N)

        eig_total = 0.0
        lan_total = 0.0
        spk_total = 0.0
        correct = 0

        for s in 1:SampleNbr
            randn!(X)
            mul!(X, sqrtΣ, X)
            X .= X / sqrt(M)
            mul!(W, X, X')
            t1, t2, sn = effprocess!(W)
            eig_total += t1
            lan_total += t2
            spk_total += sn
            correct += (sn == 3)
        end

        EigTime[m, ℓ]  = eig_total / SampleNbr
        LanTime[m, ℓ]  = lan_total / SampleNbr
        SpikeNbr[m, ℓ] = spk_total / SampleNbr
        Percent[m, ℓ]  = correct / SampleNbr
    end
end

for m in 1:len_d
    d = dvec[m]
    dlabel = lpad(Int(round(10d)), 2, '0')
    CSV.write(joinpath(tbdir, "Prct$(dlabel).csv"),
        DataFrame(N = Nvec, Percent = Percent[m, :])
    )
    CSV.write(joinpath(tbdir, "Avrg$(dlabel).csv"),
        DataFrame(N = Nvec, Spikes = SpikeNbr[m, :])
    )
    CSV.write(joinpath(tbdir, "LanTime$(dlabel).csv"),
        DataFrame(N = Nvec, LanTime = LanTime[m, :])
    )
    CSV.write(joinpath(tbdir, "EigTime$(dlabel).csv"),
        DataFrame(N = Nvec, EigTime = EigTime[m, :])
    )
end

p = plot(Nvec,Percent[1,:],color=:red,linewidth=3,label="",xlabel="N",ylabel="Probability of correct estimation",legend=:bottomright,framestyle=:box, legendfontsize=12, xtickfontsize=12, ytickfontsize=12)
p = scatter!(Nvec, Percent[1,:], markersize=4, color=:red, marker=:diamond, label="d="*string(dvec[1]))
p = plot!(Nvec,Percent[2,:],color=:blue,linewidth=3,label="")
p = scatter!(Nvec, Percent[2,:], markersize=4, color=:blue, marker=:square, label="d="*string(dvec[2]))
p = plot!(Nvec,Percent[3,:],color=:green,linewidth=3,label="")
p = scatter!(Nvec, Percent[3,:], markersize=4, color=:green, marker=:circ, label="d="*string(dvec[3]))
savefig(p,joinpath(figdir, "Prct.pdf"))

p = plot(Nvec,EigTime[2,:],color=:red,linewidth=2,label="",xlabel="N",ylabel="Time (in seconds)",legend=:topleft,framestyle=:box, legendfontsize=12, xtickfontsize=12, ytickfontsize=12)
p = scatter!(Nvec, EigTime[2,:], markersize=4, color=:red, marker=:diamond, label="Eigenvalue Computation")
p = plot!(Nvec,LanTime[2,:],color=:orange,linewidth=2,label="")
p = scatter!(Nvec, LanTime[2,:], markersize=4, color=:orange, marker=:square, label="Lanczos Approach")
savefig(p,joinpath(figdir, "Time05.pdf"))

"/Users/user/Library/CloudStorage/OneDrive-UW/Research/UW/FastSpikeDetection/Ex1Time05.pdf"

In [ ]:
Random.seed!(1234)

Nvec = vcat(200:200:3000,3500:500:8000)
len_N = length(Nvec)
dvec = [0.1,0.5,0.9]
len_d = length(dvec)

SampleNbr = 50
SuppErr_mean = zeros(len_d, len_N)
SuppErr_std  = zeros(len_d, len_N)
hErr_mean    = zeros(len_d, len_N)
hErr_std     = zeros(len_d, len_N)

function Errprocess!(W, d)
    N = size(W,1)
    results = AsympSCM(W; vecNbr=100, nx=1000,
        tol = 3/sqrt(N), errplot=true)

    γmin, γplus = results[:γmin], results[:γplus]
    xvec = results[:x]
    yvec = results[:density]

    true_γplus = 1.5 * (1 + sqrt(d))^2
    true_γmin = 1.5 * (1 - sqrt(d))^2
    supp_err = max(abs(true_γmin - γmin), abs(true_γplus - γplus))

    MP_h(x) = (true_γmin < x < true_γplus) ?
        1 / (1.5 * 2π * d * x) : 0
    hvec = yvec ./ (sqrt.(γplus .- xvec) .* sqrt.(xvec .- γmin))
    h_true = MP_h.(xvec)
    h_err = maximum(abs.(hvec .- h_true))

    return supp_err, h_err
end

N0 = first(Nvec)
d0 = first(dvec)
M0 = convert(Int64,ceil(N0/d0))
σ0 = 1.5*ones(N0)
σ0[1:3] .= (4.5, 5.0, 5.0)
sqrtΣ0 = Diagonal(sqrt.(σ0))
X0 = randn(N0, M0)
W0 = Matrix{Float64}(undef, N0, N0)
mul!(X0, sqrtΣ0, X0)
X0 .= X0 / sqrt(M0)
mul!(W0, X0, X0')
Errprocess!(W0, d0)

for m in 1:len_d
    d = dvec[m]
    for ℓ in 1:len_N
        N = Nvec[ℓ]
        M = convert(Int64,ceil(N/d))
        σvec = 1.5*ones(N)
        σvec[1:3] .= (4.5, 5.0, 5.0)
        sqrtΣ = Diagonal(sqrt.(σvec))

        X = randn(N, M)
        W = Matrix{Float64}(undef, N, N)

        supp_e = Vector{Float64}(undef, SampleNbr)
        h_e    = Vector{Float64}(undef, SampleNbr)

        for s in 1:SampleNbr
            randn!(X)
            mul!(X, sqrtΣ, X)
            X .= X / sqrt(M)
            mul!(W, X, X')
            e1, e2 = Errprocess!(W, d)
            supp_e[s] = e1
            h_e[s]    = e2
        end

        SuppErr_mean[m, ℓ] = mean(supp_e)
        SuppErr_std[m, ℓ]  = std(supp_e)
        hErr_mean[m, ℓ]    = mean(h_e)
        hErr_std[m, ℓ]     = std(h_e)
    end
end

tb_supp = DataFrame(
    N = Nvec,
    d01_mean = SuppErr_mean[1, :],
    d01_std  = SuppErr_std[1, :],
    d05_mean = SuppErr_mean[2, :],
    d05_std  = SuppErr_std[2, :],
    d09_mean = SuppErr_mean[3, :],
    d09_std  = SuppErr_std[3, :]
)
CSV.write(joinpath(tbdir, "SuppErr.csv"), tb_supp)

tb_dens = DataFrame(
    N = Nvec,
    d01_mean = hErr_mean[1, :],
    d01_std  = hErr_std[1, :],
    d05_mean = hErr_mean[2, :],
    d05_std  = hErr_std[2, :],
    d09_mean = hErr_mean[3, :],
    d09_std  = hErr_std[3, :]
)
CSV.write(joinpath(tbdir, "DensErr.csv"), tb_dens)

checkpt = ceil(Int, len_N/4)
p = plot(Nvec, (SuppErr_mean[3, checkpt] * (Nvec[checkpt])^(1/2)) * (Nvec).^(-1/2), color=:orange, linewidth=3, label=L"\mathbf{N}^{-1/2}", xlabel="N", ylabel="Errors in support", legend=:topright, yscale=:log10, linestyle=:dash, framestyle=:box, legendfontsize=12, xtickfontsize=12, ytickfontsize=12)
for (i, dcolor, dmarker) in zip(1:3, [:red, :blue, :green], [:diamond, :square, :circle])
    plot!(Nvec, SuppErr_mean[i, :], yerror=SuppErr_std[i, :], color=dcolor, linewidth=2, label="")
    scatter!(Nvec, SuppErr_mean[i, :], markersize=4, color=dcolor, marker=dmarker, label="c=$(dvec[i])")
end
savefig(p,joinpath(figdir, "SuppErr.pdf"))

p = plot(Nvec, (hErr_mean[1, checkpt] * (Nvec[checkpt])^(1/2)) * (Nvec).^(-1/2), color=:orange, linewidth=3, label=L"\mathbf{N}^{-1/2}", xlabel="N", ylabel="Errors in density", legend=:topright, yscale=:log10, linestyle=:dash, framestyle=:box, legendfontsize=12, xtickfontsize=12, ytickfontsize=12)
for (i, dcolor, dmarker) in zip(1:3, [:red, :blue, :green], [:diamond, :square, :circle])
    plot!(Nvec, hErr_mean[i, :], yerror=hErr_std[i, :], color=dcolor, linewidth=2, label="")
    scatter!(Nvec, hErr_mean[i, :], markersize=4, color=dcolor, marker=dmarker, label="c=$(dvec[i])")
end
savefig(p,joinpath(figdir, "DensErr.pdf"))

"/Users/user/Library/CloudStorage/OneDrive-UW/Research/UW/FastSpikeDetection/DensErr.pdf"